# Spark Silver Processing

This notebook is designed to be run **headlessly** by Airflow via `jupyter nbconvert --execute`.
It reads Bronze data (schedule, matchsheet, lineup) from MinIO and builds Silver Iceberg tables.

**Tables produced:**
- `lake.analytics.teams` (dimension)
- `lake.analytics.players` (dimension)
- `lake.analytics.match_statistics` (fact)
- `lake.analytics.player_match_stats` (fact)

**Do not add extraction or upload cells here.** The DAG handles that separately.

**Gold aggregation (`player_season_stats`) is handled by `spark_gold_processing.ipynb`.**

In [ ]:
# Initialize the Spark session with AQE optimizations
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

# The Spark session uses config from spark-defaults.conf (Iceberg + MinIO)
spark = (SparkSession.builder
    .appName("BrasileiraoSilverProcessing")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.adaptive.skewJoin.enabled", "true")
    .getOrCreate())

# Reduce log noise during headless execution
spark.sparkContext.setLogLevel("WARN")
print("Spark session created.")

In [ ]:
# Configuration: Bronze data location in MinIO
RAW_BUCKET = "datalake-raw"
SEASON = 2024
BRONZE_URI = f"s3a://{RAW_BUCKET}/espn/brasileirao/{SEASON}"

# Read the three Bronze JSONs from MinIO
df_schedule = spark.read.option("multiline", "true").json(f"{BRONZE_URI}/schedule.json")
df_matchsheet = spark.read.option("multiline", "true").json(f"{BRONZE_URI}/matchsheet.json")
df_lineup = spark.read.option("multiline", "true").json(f"{BRONZE_URI}/lineup.json")

print(f"Schedule:   {df_schedule.count()} rows")
print(f"Matchsheet: {df_matchsheet.count()} rows")
print(f"Lineup:     {df_lineup.count()} rows")

In [ ]:
# ==========================================================================
# DIMENSION: teams
# Unique team records with venue and capacity from matchsheet data.
# ==========================================================================

# Home teams carry venue/capacity info in matchsheet
df_home_teams = (
    df_matchsheet
    .filter(F.col("is_home") == True)
    .select(
        F.col("team").alias("team_name"),
        F.col("league"),
        F.col("season"),
        F.col("venue"),
        F.col("attendance").cast("int"),
        F.col("capacity").cast("int"),
    )
)

# For each team, pick the most frequent venue and max capacity
df_teams = (
    df_home_teams
    .groupBy("team_name", "league", "season")
    .agg(
        # Most common venue (mode): first after sorting by count desc
        F.first("venue").alias("home_venue"),
        F.max("capacity").alias("stadium_capacity"),
        F.round(F.avg("attendance"), 0).cast("int").alias("avg_attendance"),
        F.count("*").alias("home_matches"),
    )
    .withColumn("processed_at", F.current_timestamp())
)

print(f"Teams dimension: {df_teams.count()} rows")
df_teams.show(truncate=False)

In [ ]:
# ==========================================================================
# DIMENSION: players
# Unique player records derived from lineup data.
# Deduplicated by (player, team), keeping the latest known position.
# ==========================================================================

# Window to pick latest record per player+team (by game date)
player_window = Window.partitionBy("player", "team").orderBy(F.desc("game"))

df_players = (
    df_lineup
    # Rank to deduplicate: keep latest appearance per player+team
    .withColumn("rn", F.row_number().over(player_window))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .select(
        F.col("player"),
        F.col("team"),
        F.col("position"),
        F.col("league"),
        F.col("season"),
    )
)

# Add match count per player+team from the full lineup
df_player_match_count = (
    df_lineup
    .groupBy("player", "team")
    .agg(F.countDistinct("game").alias("matches_played"))
)

# Join match count into the dimension
df_players = (
    df_players
    .join(df_player_match_count, on=["player", "team"], how="left")
    .withColumn("processed_at", F.current_timestamp())
)

print(f"Players dimension: {df_players.count()} rows")
df_players.orderBy(F.desc("matches_played")).show(10, truncate=False)

In [ ]:
# ==========================================================================
# FACT: match_statistics
# Enriched match results joining schedule + derived scores + team stats.
# ==========================================================================

# --- Step 1: Derive scores from lineup (sum total_goals per game + team) ---
df_scores = (
    df_lineup
    # Cast total_goals to integer (may be null or string)
    .withColumn("total_goals", F.coalesce(F.col("total_goals").cast("int"), F.lit(0)))
    # Aggregate: sum goals per game + is_home
    .groupBy("game", "is_home")
    .agg(F.sum("total_goals").alias("team_goals"))
)

# Pivot to get home_score and away_score columns per game
df_home_scores = df_scores.filter(F.col("is_home") == True).select(
    F.col("game"), F.col("team_goals").alias("home_score")
)
df_away_scores = df_scores.filter(F.col("is_home") == False).select(
    F.col("game"), F.col("team_goals").alias("away_score")
)

# --- Step 2: Pivot matchsheet to get home/away stats side by side ---
# Stat columns from matchsheet (exclude metadata)
meta_cols = {"league", "season", "game", "team", "is_home", "venue", "attendance", "capacity"}
stat_cols = [c for c in df_matchsheet.columns if c not in meta_cols]

# Home team stats (prefix with home_)
df_home_stats = df_matchsheet.filter(F.col("is_home") == True)
for col_name in stat_cols:
    df_home_stats = df_home_stats.withColumnRenamed(col_name, f"home_{col_name}")
df_home_stats = df_home_stats.select(
    "game", "venue", "attendance", "capacity",
    *[f"home_{c}" for c in stat_cols]
)

# Away team stats (prefix with away_)
df_away_stats = df_matchsheet.filter(F.col("is_home") == False)
for col_name in stat_cols:
    df_away_stats = df_away_stats.withColumnRenamed(col_name, f"away_{col_name}")
df_away_stats = df_away_stats.select(
    "game",
    *[f"away_{c}" for c in stat_cols]
)

# --- Step 3: Join everything into one enriched match table ---
df_match_stats = (
    df_schedule
    # Join home scores
    .join(df_home_scores, on="game", how="left")
    # Join away scores
    .join(df_away_scores, on="game", how="left")
    # Join home team stats + venue/attendance
    .join(df_home_stats, on="game", how="left")
    # Join away team stats
    .join(df_away_stats, on="game", how="left")
)

# --- Step 4: Add computed analytics columns ---
df_match_stats = (
    df_match_stats
    .withColumn("home_score", F.col("home_score").cast("int"))
    .withColumn("away_score", F.col("away_score").cast("int"))
    .withColumn("total_goals", F.col("home_score") + F.col("away_score"))
    .withColumn("goal_diff", F.abs(F.col("home_score") - F.col("away_score")))
    .withColumn("is_draw", F.col("home_score") == F.col("away_score"))
    .withColumn("winner", F.when(
        F.col("home_score") > F.col("away_score"), F.col("home_team")
    ).when(
        F.col("away_score") > F.col("home_score"), F.col("away_team")
    ).otherwise(F.lit("Draw")))
    .withColumn("processed_at", F.current_timestamp())
)

print(f"Match statistics: {df_match_stats.count()} rows, {len(df_match_stats.columns)} columns")

In [ ]:
# ==========================================================================
# FACT: player_match_stats
# Per-match player performance from lineup data (one row per player per game).
# ==========================================================================

numeric_cols = [
    "total_goals", "goal_assists", "shots_on_target", "total_shots",
    "fouls_committed", "fouls_suffered", "yellow_cards", "red_cards",
    "own_goals", "offsides", "appearances", "saves", "shots_faced",
    "goals_conceded"
]

# Cast numeric columns, filling nulls with 0
df_player_match = df_lineup
for col_name in numeric_cols:
    if col_name in df_player_match.columns:
        df_player_match = df_player_match.withColumn(
            col_name,
            F.coalesce(F.col(col_name).cast("int"), F.lit(0))
        )

# ESPN defines starters. In earlier dumps it might be is_starter.
if "starter" not in df_player_match.columns and "is_starter" in df_player_match.columns:
    df_player_match = df_player_match.withColumn("starter", F.col("is_starter") == True)
elif "starter" not in df_player_match.columns and "formation_place" in df_player_match.columns:
    df_player_match = df_player_match.withColumn("starter", F.col("formation_place").isNotNull())
else:
    # Fallback if the field does not exist at all
    df_player_match = df_player_match.withColumn("starter", F.lit(True))

# Add processing timestamp
df_player_match = df_player_match.withColumn("processed_at", F.current_timestamp())

print(f"Player match stats: {df_player_match.count()} rows")

In [ ]:
# ==========================================================================
# WRITE: Persist all Silver tables to Iceberg
# ==========================================================================

spark.sql("CREATE NAMESPACE IF NOT EXISTS lake.analytics")

# Silver dimensions
df_teams.writeTo("lake.analytics.teams").createOrReplace()
print("Wrote lake.analytics.teams")

df_players.writeTo("lake.analytics.players").createOrReplace()
print("Wrote lake.analytics.players")

# Silver facts
df_match_stats.writeTo("lake.analytics.match_statistics").createOrReplace()
print("Wrote lake.analytics.match_statistics")

df_player_match.writeTo("lake.analytics.player_match_stats").createOrReplace()
print("Wrote lake.analytics.player_match_stats")

print("\nAll 4 Silver Iceberg tables written successfully.")

In [ ]:
# ==========================================================================
# VERIFY: Quick row counts for all tables
# ==========================================================================

tables = [
    "lake.analytics.teams",
    "lake.analytics.players",
    "lake.analytics.match_statistics",
    "lake.analytics.player_match_stats",
]

print("Table verification:")
for t in tables:
    cnt = spark.sql(f"SELECT COUNT(*) as cnt FROM {t}").collect()[0]["cnt"]
    print(f"  {t}: {cnt} rows")

In [ ]:
# Stop the Spark session to free all memory
spark.stop()
print("Spark session stopped. Memory released.")